In [245]:
import json
import os
import io
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import zipfile

# **1. Load data & setup config**

In [246]:
# Load data
with zipfile.ZipFile("../csv_outputs/feature_df.zip") as zf:
    with zf.open("feature_df.csv") as f:
        hist_feature_df = pd.read_csv(f)
df_historical = pd.read_csv("../../datasets/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
df_rpi = pd.read_csv("../../datasets/HDBResalePriceIndex1Q2009100Quarterly.csv")
output_json = "../json_outputs/listings_predictions.json"
model_paths = {
    "lgbm": "../models/lgb_model.joblib",
    "xgb": "../models/xgb_model.ubj",
    "cb": "../models/cb_model.cbm"
}
ci_offsets_json = "../json_outputs/ci_offsets.json"

SELECTED_MODEL = "ensemble_equal" # lgbm/xgb/cb/ensemble/ensemble_equal - we choose the best-performing model as per the DM tests in model_training.ipynb
ENSEMBLE_WEIGHTS = None

# Define constants
RPI_BASE = 100.0
FEATURES = [
    "town", "flat_type", "floor_area_sqm", "lease_commence_date",
    "remaining_lease", "lat", "lon",
    "mall_1_dist_m", "mall_2_dist_m", "mall_3_dist_m",
    "school_1_dist_m", "school_2_dist_m", "school_3_dist_m",
    "hawker_1_dist_m", "hawker_2_dist_m", "hawker_3_dist_m",
    "polyclinic_1_dist_m", "polyclinic_2_dist_m", "polyclinic_3_dist_m",
    "supermarket_1_dist_m", "supermarket_2_dist_m", "supermarket_3_dist_m",
    "train_1_dist_m", "train_2_dist_m", "train_3_dist_m",
    "bus_1_dist_m", "bus_2_dist_m", "bus_3_dist_m",
    "storey_midpoint",
    "num_mrt_within_1km", "flag_mrt_within_500m",
    "num_primary_schools_within_1km", "num_hawkers_within_500m",
    "num_bus_within_400m",
    "dist_cbd",
    "month_index"
]

In [247]:
# RPI: get current quarter value
df_rpi.rename(columns={"index": "rpi"}, inplace=True)
df_rpi["rpi"] = pd.to_numeric(df_rpi["rpi"])
df_rpi.loc[len(df_rpi)] = { # Flash estimate from HDB
    "quarter": "2026-Q1",
    "rpi": 203.4
 }

today = pd.Timestamp.today()
current_quarter = f"{today.year}-Q{((today.month - 1) // 3) + 1}"

if current_quarter not in df_rpi["quarter"].values:
  print(f"Current quarter {current_quarter} not in RPI data. Extrapolating...")

  recent = df_rpi.tail(4).copy().reset_index(drop=True)
  recent["t"] = range(4)
  slope, intercept = np.polyfit(recent["t"], recent["rpi"], deg=1)

  last_quarter = df_rpi["quarter"].iloc[-1]
  lq_year, lq_num = int(last_quarter.split("-Q")[0]), int(last_quarter.split("-Q")[1])
  q_year,  q_num  = int(current_quarter.split("-Q")[0]), int(current_quarter.split("-Q")[1])
  steps_ahead = (q_year - lq_year) * 4 + (q_num - lq_num)
  t_extrap = 3 + steps_ahead
  rpi_extrap = round(intercept + slope * t_extrap, 1)

  print(f"Extrapolated RPI for {current_quarter}: {rpi_extrap:.1f} (last known: {df_rpi["rpi"].iloc[-1]:.1f}, slope: {slope:+.2f}/quarter)")
  df_rpi = pd.concat(
      [df_rpi, pd.DataFrame([{"quarter": current_quarter, "rpi": rpi_extrap}])],
      ignore_index=True
  )
else:
  val = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
  print(f"RPI for {current_quarter}: {val:.1f}")

RPI_BASE = 100.0
RPI_CURRENT = df_rpi.loc[df_rpi["quarter"] == current_quarter, "rpi"].iloc[0]
print(f"Base RPI: {RPI_BASE}")
print(f"Current RPI: {RPI_CURRENT:.1f} (used to convert predictions back to nominal value)")

Current quarter 2026-Q2 not in RPI data. Extrapolating...
Extrapolated RPI for 2026-Q2: 203.8 (last known: 203.4, slope: +0.14/quarter)
Base RPI: 100.0
Current RPI: 203.8 (used to convert predictions back to nominal value)


In [248]:
# Get listings data (2026 onwards)
df_listings = hist_feature_df[hist_feature_df["month_index"] >= 108].reset_index(drop=True)
df_listings["rpi"] = RPI_CURRENT # since we treat 2026 data as "current listings", we use the extrapolated RPI for the current quarter (2026-Q2)
for col in ["town", "flat_type"]:
    df_listings[col] = df_listings[col].astype("category")
df_listings

,town,flat_type,block,street_name,floor_area_sqm,lease_commence_date,remaining_lease,lat,lon,postal,...,storey_midpoint,num_mrt_within_1km,flag_mrt_within_500m,num_primary_schools_within_1km,num_hawkers_within_500m,num_bus_within_400m,dist_cbd,rpi,real_price,month_index
0,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,1978,51.083,1.366227,103.850086,560314,...,11.0,1,1,3,1,7,9130.547015,203.8,169616.519174,108
1,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,44.0,1978,51.000,1.366227,103.850086,560314,...,11.0,1,1,3,1,7,9130.547015,203.8,159783.677483,109
2,ANG MO KIO,2 ROOM,508,ANG MO KIO AVE 8,44.0,1980,53.333,1.373790,103.849029,560508,...,5.0,2,1,3,1,10,9973.901271,203.8,162241.887906,110
3,ANG MO KIO,3 ROOM,308B,ANG MO KIO AVE 1,70.0,2012,85.750,1.365266,103.844538,562308,...,5.0,1,0,2,1,4,9055.184826,203.8,323500.491642,108
4,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,73.0,1976,49.333,1.366558,103.841624,680215,...,11.0,2,0,1,2,7,9231.105024,203.8,213864.306785,108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6053,YISHUN,EXECUTIVE,643,YISHUN ST 61,142.0,1987,60.500,1.421335,103.837437,760643,...,11.0,2,0,4,0,12,15336.454532,203.8,410521.140610,110
6054,YISHUN,EXECUTIVE,611,YISHUN ST 61,146.0,1987,60.750,1.420201,103.836153,760611,...,8.0,1,1,3,0,13,15226.277355,203.8,436578.171091,110
6055,YISHUN,EXECUTIVE,877,YISHUN ST 81,142.0,1987,60.833,1.413902,103.835454,760877,...,11.0,1,1,2,0,7,14539.801915,203.8,481809.242871,110
6056,YISHUN,EXECUTIVE,836,YISHUN ST 81,146.0,1988,61.000,1.415452,103.833091,760836,...,11.0,1,1,2,0,10,14744.951102,203.8,489183.874140,110


# **2. Preparing outputs**
* Run models to get `predicted_price`
* Get confidence intervals of `predicted_price`: `ci_lower` and `ci_upper`
* Get recent median price of **pre-2026** transactions: `median_similar`
  * default: take last 6 months
  * if < 20 transactions of the same (town, flat_type) combination within the last 6 months, expand the window
  * if we expand the window beyond the last 12 months, flag the median as old / "stale" 
  * count the number of months we go back by
  * some (town, flat_type) combinations are not represented at all in historical data --> returns NA 
* Get `valuation_pct` (predicted vs actual resale price)

In [249]:
# Load models
def load_models(model_name):
  models = {}
  if model_name in ("lgbm", "ensemble", "ensemble_equal"):
    print("Loading LightGBM model")
    with zipfile.ZipFile("../models/lgb_model.zip") as zf:
      with zf.open("lgb_model.joblib") as f:
        models["lgbm"] = joblib.load(io.BytesIO(f.read()))
  if model_name in ("xgb", "ensemble", "ensemble_equal"):
    print("Loading XGBoost model")
    m = xgb.XGBRegressor()
    m.load_model(model_paths["xgb"])
    models["xgb"] = m
  if model_name in ("cb", "ensemble", "ensemble_equal"):
    print("Loading CatBoost model")
    m = CatBoostRegressor()
    m.load_model(model_paths["cb"])
    models["cb"] = m
  return models

def predict(models, X):
  cat_cols = ["town", "flat_type"]

  def cb_predict(X):
    X_cb = X.copy()
    for col in cat_cols:
      X_cb[col] = X_cb[col].astype(str)
    pool = Pool(X_cb, cat_features=cat_cols)
    return models["cb"].predict(pool)

  if SELECTED_MODEL == "ensemble":
    w = ENSEMBLE_WEIGHTS
    if w is None:
      weights_path = "../models/ensemble_weights.npy"
      if os.path.exists(weights_path):
        w = np.load(weights_path)
      else:
        n = len(models)
        w = np.ones(n) / n
        print(f"Warning: ensemble_weights.npy not found at {weights_path}. Using equal weights {w}.")
    return (
        w[0] * cb_predict(X)
        + w[1] * models["xgb"].predict(X)
        + w[2] * models["lgbm"].predict(X)
    )
  
  if SELECTED_MODEL == "ensemble_equal":
    return np.stack([
        cb_predict(X),
        models["xgb"].predict(X),
        models["lgbm"].predict(X),
    ], axis=1).mean(axis=1)
  if SELECTED_MODEL == "cb":
    return np.array(cb_predict(X), dtype=float)
  return np.array(models[SELECTED_MODEL].predict(X), dtype=float)

models = load_models(SELECTED_MODEL)

Loading LightGBM model


Loading XGBoost model
Loading CatBoost model


In [250]:
# Predicted price
print(f"Loading model(s): {SELECTED_MODEL}")
X = df_listings[FEATURES].copy()
for col in ["town", "flat_type"]:
  X[col] = X[col].astype("category")

print(f"Running predictions on {len(X)} listings")
pred_real = predict(models, X)
pred_nominal = pred_real * (df_listings["rpi"].values / RPI_BASE)
print(f"Predicted {len(pred_real)} listings")

Loading model(s): ensemble_equal
Running predictions on 6058 listings
Predicted 6058 listings


In [251]:
# Confidence intervals
ci_lower_arr = ci_upper_arr = None
if os.path.exists(ci_offsets_json):
  with open(ci_offsets_json) as f:
    all_offsets = json.load(f)
  offsets = all_offsets.get(SELECTED_MODEL) or all_offsets.get("ensemble_equal")
  if offsets:
    rpi_vals = df_listings["rpi"].values / RPI_BASE
    ci_lower_arr = (pred_real + offsets["p025_real"]) * rpi_vals
    ci_upper_arr = (pred_real + offsets["p975_real"]) * rpi_vals
    key_used = SELECTED_MODEL if SELECTED_MODEL in all_offsets else "ensemble_equal"
    print(f"CI offsets applied from key {key_used}.")
else:
  print("ci_offsets.json not found, CI columns will be null.")

df_listings["pred_price_lower"] = ci_lower_arr if ci_lower_arr is not None else None
df_listings["pred_price_upper"] = ci_upper_arr if ci_upper_arr is not None else None

CI offsets applied from key ensemble_equal.


In [252]:
# Recent median (using December 2025 as the reference point)
MIN_SAMPLE = 20 
DEFAULT_MONTHS = 6

def compute_recent_median(historical, min_sample=MIN_SAMPLE, default_months=DEFAULT_MONTHS):
    df = historical.copy()
    df["month"] = pd.to_datetime(df["month"])
    df = df.sort_values("month", ascending=False)

    # Set reference to Dec 2025 so the 6-month window is June 2025 - Dec 2025
    reference_date = pd.Timestamp("2025-12-01")
    default_cutoff = reference_date - pd.DateOffset(months=default_months)

    medians = {}
    months_back = {}
    sample_sizes = {}

    for (town, flat_type), group in df.groupby(["town", "flat_type"]):
        window = group[group["month"] >= default_cutoff]

        if len(window) >= min_sample:
            sample = window 
        else:
            sample = group.head(min_sample) 

        medians[(town, flat_type)] = sample["resale_price"].median()
        sample_sizes[(town, flat_type)] = len(sample)

        oldest = sample["month"].min()
        n_months = (reference_date.year - oldest.year) * 12 + (reference_date.month - oldest.month)
        months_back[(town, flat_type)] = n_months

    return medians, months_back, sample_sizes

df_historical_pre2026 = df_historical[df_historical["month"] < "2026-01"]
medians_recent, months_back, sample_sizes = compute_recent_median(df_historical_pre2026)

# Apply to listings
df_listings["median_similar"] = df_listings.apply(lambda r: medians_recent.get((r["town"], r["flat_type"])), axis=1)
df_listings["median_months_back"] = df_listings.apply(lambda r: months_back.get((r["town"], r["flat_type"])), axis=1)
df_listings["median_sample_size"] = df_listings.apply(lambda r: sample_sizes.get((r["town"], r["flat_type"])), axis=1)
df_listings["median_old"] = df_listings["median_months_back"].apply(lambda x: bool(x > 12) if pd.notna(x) else False)          

In [253]:
# Valuation (actual resale price treated as asking price)
df_listings["asking_price"] = (df_listings["real_price"] * df_listings["rpi"] / RPI_BASE).round(0)
df_listings["predicted_price"] = pred_nominal.round(0)
df_listings["valuation_pct"] = (
    (df_listings["asking_price"] - df_listings["predicted_price"]) / df_listings["predicted_price"] * 100
).round(2)

In [254]:
# Assemble and write output
records = []
for i, row in df_listings.iterrows():
  record = {
      "town": row.get("town"),
      "flat_type": row.get("flat_type"),
      "floor_area_sqm": (float(row["floor_area_sqm"]) if pd.notna(row.get("floor_area_sqm")) else None),
      "lease_commence_date": (int(row["lease_commence_date"]) if pd.notna(row.get("lease_commence_date")) else None),
      "remaining_lease": (float(row["remaining_lease"]) if pd.notna(row.get("remaining_lease")) else None),
      "storey_midpoint": (float(row["storey_midpoint"]) if pd.notna(row.get("storey_midpoint")) else None),
      "block": row.get("block"),
      "street_name": row.get("street_name"),
      "full_address": row.get("full_address"),
      "postal": (str(int(row["postal"])) if pd.notna(row.get("postal")) else None),
      "lat": float(row["lat"]),
      "lon": float(row["lon"]),
      "month_index": (int(row["month_index"]) if pd.notna(row.get("month_index")) else None),
      "asking_price": (float(row["asking_price"]) if pd.notna(row.get("asking_price")) else None),
      "predicted_price": float(pred_nominal[i]),
      "predicted_price_lower": (float(row["pred_price_lower"]) if pd.notna(row.get("pred_price_lower")) else None),
      "predicted_price_upper": (float(row["pred_price_upper"]) if pd.notna(row.get("pred_price_upper")) else None),
      "valuation_pct": (float(row["valuation_pct"]) if pd.notna(row.get("valuation_pct")) else None),
      "median_similar": (float(row["median_similar"]) if pd.notna(row.get("median_similar")) else None),
      "median_months_back": (int(row["median_months_back"]) if pd.notna(row.get("median_months_back")) else None),
      "median_sample_size": (int(row["median_sample_size"]) if pd.notna(row.get("median_sample_size")) else None),
      "median_old": row.get("median_old")
  }
  records.append(record)

with open(output_json, "w") as f:
  json.dump(records, f, indent=2, default=str)
print(f"Wrote {len(records)} records to {output_json}")

## Check
for r in records[:3]:
  print(f"town={r['town']} flat_type={r['flat_type']} | pred = S${r['predicted_price']:,.0f} | actual = S${r['asking_price']:,.0f} | val_pct = {r['valuation_pct']}")

Wrote 6058 records to ../json_outputs/listings_predictions.json
town=ANG MO KIO flat_type=2 ROOM | pred = S$344,699 | actual = S$345,678 | val_pct = 0.28
town=ANG MO KIO flat_type=2 ROOM | pred = S$344,699 | actual = S$325,639 | val_pct = -5.53
town=ANG MO KIO flat_type=2 ROOM | pred = S$332,580 | actual = S$330,649 | val_pct = -0.58


In [257]:
df_listings.to_csv("../csv_outputs/listings_predictions.csv", index=0)

In [258]:
# Diagnostics/checking

## Predictions vs actual
over = (pred_nominal > df_listings["asking_price"].values).sum()
total = len(pred_nominal)
print(f"Predicted > actual: {over}/{total} ({over/total*100:.1f}%)")
under = (pred_nominal < df_listings["asking_price"].values).sum()
print(f"Actual > predicted: {under}/{total} ({under/total*100:.1f}%)")

## Range of valuation_pct
print(df_listings["valuation_pct"].describe())

## Checking median transacted price
print(sum(df_listings["median_similar"].isna()))
print(df_listings[df_listings["median_similar"].isna()][["town", "flat_type"]].value_counts())

## Checking columns with NAs
print(df_listings.isna().sum()[df_listings.isna().sum() > 0])

Predicted > actual: 4602/6058 (76.0%)
Actual > predicted: 1456/6058 (24.0%)
count    6058.000000
mean       -3.215842
std         5.316821
min       -39.660000
25%        -6.257500
50%        -3.235000
75%        -0.172500
max        34.010000
Name: valuation_pct, dtype: float64
0
town        flat_type       
ANG MO KIO  1 ROOM              0
            2 ROOM              0
            3 ROOM              0
            4 ROOM              0
            5 ROOM              0
                               ..
YISHUN      3 ROOM              0
            4 ROOM              0
            5 ROOM              0
            EXECUTIVE           0
            MULTI-GENERATION    0
Name: count, Length: 182, dtype: int64
Series([], dtype: int64)
